In [2]:
import sqlite3
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gzip
from tqdm.notebook import tqdm
import gc

plt.style.use("ggplot")

In [3]:
# Connect to database
conn = sqlite3.connect("mimic.db")
conn.execute("PRAGMA foreign_keys = ON;")

# Confirm connection
pd.read_sql("PRAGMA database_list;", conn)

,seq,name,file
0,0,main,/blue/cap5771/shesadree.p/CAP5771_lili1501/mim...


In [4]:
import pandas as pd

pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn)

,name


In [5]:
schema = """
CREATE TABLE patients (
    subject_id INTEGER PRIMARY KEY,
    gender TEXT,
    anchor_age INTEGER,
    anchor_year INTEGER,
    anchor_year_group TEXT,
    dod TEXT
);

CREATE TABLE admissions (
    hadm_id INTEGER PRIMARY KEY,
    subject_id INTEGER,
    admittime TEXT,
    dischtime TEXT,
    deathtime TEXT,
    admission_type TEXT,
    admit_provider_id TEXT,
    admission_location TEXT,
    discharge_location TEXT,
    insurance TEXT,
    language TEXT,
    marital_status TEXT,
    race TEXT,
    edregtime TEXT,
    edouttime TEXT,
    hospital_expire_flag INTEGER,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id)
);

CREATE TABLE diagnoses_icd (
    subject_id INTEGER,
    hadm_id INTEGER,
    seq_num INTEGER,
    icd_code TEXT,
    icd_version INTEGER,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id)
);



CREATE TABLE labevents (
    labevent_id INTEGER NOT NULL,
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    specimen_id INTEGER NOT NULL,
    itemid INTEGER NOT NULL,
    order_provider_id TEXT,
    charttime TEXT,
    storetime TEXT,
    value TEXT,
    valuenum REAL,
    valueuom TEXT,
    ref_range_lower REAL,
    ref_range_upper REAL,
    flag TEXT,
    priority TEXT,
    comments TEXT,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id)
);
"""
conn.executescript(schema)


In [6]:
schema = """

CREATE TABLE d_icd_diagnoses (
    icd_code TEXT NOT NULL,
    icd_version INTEGER NOT NULL,
    long_title TEXT,
    PRIMARY KEY (icd_code, icd_version)
);


CREATE TABLE d_icd_procedures (
    icd_code TEXT NOT NULL,
    icd_version INTEGER NOT NULL,
    long_title TEXT,
    PRIMARY KEY (icd_code, icd_version)
);


CREATE TABLE d_labitems (
    itemid INTEGER PRIMARY KEY,
    label TEXT,
    fluid TEXT,
    category TEXT
);


CREATE TABLE d_items (
    itemid INTEGER PRIMARY KEY,
    label TEXT,
    abbreviation TEXT,
    linksto TEXT,
    category TEXT,
    unitname TEXT,
    param_type TEXT,
    lownormalvalue REAL,
    highnormalvalue REAL
);

"""
conn.executescript(schema)


In [7]:
schema = """

CREATE TABLE icustays (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER NOT NULL,
    first_careunit TEXT,
    last_careunit TEXT,
    intime TEXT,
    outtime TEXT,
    los REAL,
    PRIMARY KEY (stay_id)
);


CREATE TABLE inputevents (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER,
    caregiver_id INTEGER,
    starttime TEXT,
    endtime TEXT,
    storetime TEXT,
    itemid INTEGER,
    amount REAL,
    amountuom TEXT,
    rate REAL,
    rateuom TEXT,
    orderid INTEGER,
    linkorderid INTEGER,
    ordercategoryname TEXT,
    secondaryordercategoryname TEXT,
    ordercomponenttypedescription TEXT,
    ordercategorydescription TEXT,
    patientweight REAL,
    totalamount REAL,
    totalamountuom TEXT,
    isopenbag INTEGER,
    continueinnextdept INTEGER,
    statusdescription TEXT,
    originalamount REAL,
    originalrate REAL
);


CREATE TABLE outputevents (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER,
    caregiver_id INTEGER,
    charttime TEXT,
    storetime TEXT,
    itemid INTEGER,
    value REAL,
    valueuom TEXT
);



CREATE TABLE chartevents (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER,
    caregiver_id INTEGER,
    charttime TEXT,
    storetime TEXT,
    itemid INTEGER,
    value TEXT,
    valuenum REAL,
    valueuom TEXT,
    warning INTEGER
);


"""
conn.executescript(schema)


In [8]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';",conn)

,name
0,patients
1,admissions
2,diagnoses_icd
3,labevents
4,d_icd_diagnoses
5,d_icd_procedures
6,d_labitems
7,d_items
8,icustays
9,inputevents


In [11]:
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "mimic dataset"

In [12]:
def load_csv_gz(path, table, subject_set=None, item_ids=None, chunksize=None):

    print(f"Loading {path.name} → {table}")

    read_cols = None  # optional: reduce memory by selecting columns

    if chunksize:
        for chunk in tqdm(
            pd.read_csv(path, compression="gzip", chunksize=chunksize),
            desc=table
        ):

            # Filter subject_id
            if subject_set is not None and "subject_id" in chunk.columns:
                chunk = chunk[chunk["subject_id"].isin(subject_set)]

            # Filter itemid (labs or chart events)
            if item_ids is not None and "itemid" in chunk.columns:
                chunk = chunk[chunk["itemid"].isin(item_ids)]

            if not chunk.empty:
                chunk.to_sql(table, conn, if_exists="append", index=False)

            del chunk
            gc.collect()

    else:
        df = pd.read_csv(path, compression="gzip")

        if subject_set is not None and "subject_id" in df.columns:
            df = df[df["subject_id"].isin(subject_set)]

        if item_ids is not None and "itemid" in df.columns:
            df = df[df["itemid"].isin(item_ids)]

        if not df.empty:
            df.to_sql(table, conn, if_exists="replace", index=False)

In [13]:
load_csv_gz(DATA_DIR / "hosp/patients.csv.gz", "patients")

Loading patients.csv.gz → patients


In [14]:
sampled_subjects = pd.read_sql("""
SELECT subject_id
FROM patients
ORDER BY RANDOM()
LIMIT (
    SELECT CAST(COUNT(DISTINCT subject_id) * 0.5 AS INT)
    FROM patients
);
""", conn)

print(f"Sampled {len(sampled_subjects)} subjects.")
sampled_subjects.head()

Sampled 182313 subjects.


,subject_id
0,11894641
1,14704448
2,15368467
3,12895725
4,12845296


In [24]:
load_csv_gz(DATA_DIR / "hosp/d_icd_diagnoses.csv.gz", "d_icd_diagnoses")
load_csv_gz(DATA_DIR / "hosp/d_icd_procedures.csv.gz", "d_icd_procedures")
load_csv_gz(DATA_DIR / "hosp/d_labitems.csv.gz", "d_labitems")


Loading d_icd_diagnoses.csv.gz → d_icd_diagnoses
Loading d_icd_procedures.csv.gz → d_icd_procedures
Loading d_labitems.csv.gz → d_labitems


In [26]:
#icu tables
load_csv_gz(DATA_DIR / "icu/d_items.csv.gz", "d_items")

Loading d_items.csv.gz → d_items


##### charevents

In [28]:
hr_ids = pd.read_sql("""
SELECT itemid
FROM d_items
WHERE LOWER(label) = 'heart rate'
""", conn)["itemid"].tolist()

print(hr_ids)

[220045]


In [30]:
map_ids = pd.read_sql("""
SELECT itemid
FROM  d_items 
WHERE LOWER(label) IN ('arterial blood pressure mean','non invasive blood pressure mean')
""", conn)["itemid"].tolist()

print(map_ids)

[220052, 220181]


In [31]:
rr_ids = pd.read_sql("""
SELECT itemid
FROM  d_items 
WHERE LOWER(label) IN (
    'respiratory rate',
    'respiratory rate (spontaneous)',
    'respiratory rate (total)'
)
""", conn)["itemid"].tolist()

print(rr_ids)

[220210, 224689, 224690]


In [32]:
sbp_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) IN ('arterial blood pressure systolic','non invasive blood pressure systolic')
""", conn)["itemid"].tolist()

print(sbp_ids)

dbp_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) IN ('arterial blood pressure diastolic','non invasive blood pressure diastolic')
""", conn)["itemid"].tolist()

print(dbp_ids)

[220050, 220179]
[220051, 220180]


In [33]:
# oxygen saturation itemids
spo2_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) = 'o2 saturation pulseoxymetry'
""", conn)["itemid"].tolist()

print(spo2_ids)

[220277]


In [35]:
fio2_ids = pd.read_sql("""
SELECT itemid
FROM  d_items 
WHERE LOWER(label) = 'inspired o2 fraction'
""", conn)["itemid"].tolist()

print(fio2_ids)

[223835]


In [36]:
pao2_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) = 'arterial o2 pressure'
""", conn)["itemid"].tolist()

print(pao2_ids)

[220224]


In [38]:

temp_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) IN (
    'temperature fahrenheit',
    'temperature celsius'
)
""", conn)["itemid"].tolist()

print(temp_ids)

[223761, 223762]


##### labevents

In [39]:
creatinine_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) IN ('creatinine')
""", conn)["itemid"].tolist()

print(creatinine_ids)

[50912, 52546]


In [40]:
lactate_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) = 'lactate'
""", conn)["itemid"].tolist()

print(lactate_ids)

[50813, 52442, 53154]


In [41]:
bilirubin_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems 
WHERE LOWER(label) = 'bilirubin, total'
""", conn)["itemid"].tolist()

print(bilirubin_ids)

[50885, 53089]


In [42]:
wbc_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems 
WHERE LOWER(label) IN ('wbc', 'wbc count','white blood cells')
""", conn)["itemid"].tolist()

print(wbc_ids)

[51300, 51301, 51516, 51755, 51756, 52407]


In [43]:
platelet_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems 
WHERE LOWER(label) = 'platelet count'
""", conn)["itemid"].tolist()

print(platelet_ids)

[51265, 53189]


In [45]:
hemog_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) IN( 'hemoglobin', 'absolute hemoglobin','hemoglobin, calculated')
""", conn)["itemid"].tolist()

print(hemog_ids)

sodium_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems
WHERE LOWER(label) IN ('sodium', 'sodium, whole blood')
""", conn)["itemid"].tolist()

print(sodium_ids)

potassium_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) IN ('potassium', 'potassium, whole blood')
""", conn)["itemid"].tolist()

print(potassium_ids)

bun_ids = 51842

[50811, 50855, 51222, 51640, 51645]
[50824, 50983, 52455, 52623]
[50822, 50833, 50971, 52452, 52610]


In [46]:
labevent_ids = [creatinine_ids,lactate_ids,bilirubin_ids,wbc_ids,platelet_ids,
                hemog_ids,sodium_ids,potassium_ids,bun_ids]

charevent_ids = [hr_ids,rr_ids,map_ids,sbp_ids,dbp_ids,fio2_ids,pao2_ids,temp_ids]

In [47]:
labevent_ids

[[50912, 52546],
 [50813, 52442, 53154],
 [50885, 53089],
 [51300, 51301, 51516, 51755, 51756, 52407],
 [51265, 53189],
 [50811, 50855, 51222, 51640, 51645],
 [50824, 50983, 52455, 52623],
 [50822, 50833, 50971, 52452, 52610],
 51842]

In [48]:
from itertools import chain

lab_item_ids = list(
    chain.from_iterable(
        x if isinstance(x, list) else [x]
        for x in labevent_ids
    )
)


char_item_ids = list(
    chain.from_iterable(
        x if isinstance(x, list) else [x]
        for x in charevent_ids
    )
)




In [49]:
lab_item_ids = set(lab_item_ids)
char_item_ids = set(char_item_ids)
# char_item_ids

In [50]:
subject_set = set(sampled_subjects["subject_id"])

In [51]:
load_csv_gz(DATA_DIR / "hosp/admissions.csv.gz", "admissions", subject_set=subject_set)
load_csv_gz(DATA_DIR / "hosp/diagnoses_icd.csv.gz", "diagnoses_icd", subject_set=subject_set)
load_csv_gz(DATA_DIR / "icu/icustays.csv.gz", "icustays", subject_set=subject_set)

Loading admissions.csv.gz → admissions
Loading diagnoses_icd.csv.gz → diagnoses_icd
Loading icustays.csv.gz → icustays


In [ ]:
load_csv_gz(DATA_DIR / "hosp/labevents.csv.gz","labevents", subject_set=subject_set,item_ids=lab_item_ids,chunksize=50_000)
load_csv_gz(DATA_DIR / "icu/inputevents.csv.gz", "inputevents", subject_set=subject_set, chunksize=50_000)
load_csv_gz(DATA_DIR / "icu/outputevents.csv.gz", "outputevents", subject_set=subject_set, chunksize=50_000)
load_csv_gz(DATA_DIR / "icu/chartevents.csv.gz", "chartevents", subject_set=subject_set, item_ids=char_item_ids,chunksize=50_000)